# STIR-Net V1 — 27 Overnight Train + Diagnose

This notebook is designed to run **unattended for the night** on the current STIR-Net V1 repository state.

It keeps the **Notebook-24 architecture**:

```text
spatial encoder/decoder
        ↓
learned high-resolution spatial proposals
        ↓
proposal-specific query anchors
        ↓
query decoder
        ↓
native masks
```

The pre-coreasoning Notebook-25 patch is **not** used.

## Overnight questions

By morning this run should tell us:

1. Can the learned proposal mechanism converge from an overcomplete set to approximately one proposal per real cell while retaining recall?
2. Given good proposal anchors, can the existing query decoder learn useful source-9 instance masks with enough optimization?
3. Are the masks controlled mainly by **anchor geometry** or by query-content differences?
4. Does native rendering improve/degrade what is already learned at coarse resolution?
5. Does enabling the full temporal/coreasoning model help or hurt after the spatial-only mechanism has been trained?

## Robustness / disk policy

The notebook is intentionally conservative about disk use:

- one timestamped run directory;
- one rolling `checkpoint_recovery.pt`;
- at most one best checkpoint for each important stage;
- model-only checkpoints (no optimizer state);
- CSV / JSONL logs are appended continuously;
- no large full-volume prediction dumps;
- one optional compact source-9 visualization NPZ at the end.

Every long stage is wrapped in `try/except`. A stage failure is logged and the controller attempts to continue with the best surviving state.

## Time budget

Default:

```text
total wall-clock budget      7.5 h
final evaluation reserve     45 min

proposal warm-up          <= 75 min
spatial-only query train  <= 135 min
spatial-only native train <= 90 min
full joint train          = remaining training time
final diagnosis           = reserved time
```

The controller checks wall-clock time before every new step.

> Use **Run All**. The heavy work is driven by the final controller cell.


In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
from typing import Any, Callable
import copy
import csv
import gc
import json
import math
import os
import shutil
import subprocess
import time
import traceback

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.debugging.probes.matching import run_matching_probe
from learned.stirnet.model.matcher import (
    build_local_support_masks,
    target_ids,
    target_masks_at_shape,
)
from learned.stirnet.model.native_masks import compose_native_query_logits
from learned.stirnet.model.query_builder import QUERY_SPATIAL_PROPOSAL
from learned.stirnet.model.types import QueryState, StirNetOutput, TemporalState
from learned.stirnet.training.checkpoint import load_checkpoint, save_checkpoint
from learned.stirnet.training.curriculum import model_parameter_groups
from learned.stirnet.training.trainer import (
    Trainer,
    model_forward_from_batch,
    move_batch_to_device,
)

# ---------------------------------------------------------------------
# User-adjustable overnight configuration
# ---------------------------------------------------------------------

SEED = 40266
SOURCE_ID = 9

AMP_DTYPE = torch.float16
BASE_LR = 2e-4

TOTAL_BUDGET_HOURS = 7.5
EVALUATION_RESERVE_MINUTES = 45

PROPOSAL_MAX_MINUTES = 75
SPATIAL_QUERY_MAX_MINUTES = 135
SPATIAL_NATIVE_MAX_MINUTES = 90

PROPOSAL_MAX_STEPS = 60
SPATIAL_QUERY_MAX_STEPS = 120
SPATIAL_NATIVE_MAX_STEPS = 80
TEMPORAL_WARMUP_MAX_STEPS = 10
JOINT_MAX_STEPS = 80

PROPOSAL_EVAL_EVERY = 5
QUERY_EVAL_EVERY = 5
NATIVE_EVAL_EVERY = 5
JOINT_EVAL_EVERY = 3

RECOVERY_SAVE_EVERY = 5

# Gate used only to decide whether query training should start from the
# best learned proposal state.
MIN_SOURCE9_RECALL_1DREF = 8 / 9

# Final Napari-ready artifact. It stores only a small source-9 crop and
# combined predictions, never all full-volume query masks.
SAVE_COMPACT_VISUAL_ARTIFACT = True

# Keep false so Run All never opens a GUI and blocks overnight execution.
OPEN_NAPARI_AT_END = False

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

STEP30_CHECKPOINT = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "12_staged_same_sample"
    / "checkpoint_spatial_dense.pt"
)

RUN_PARENT = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "overnight"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 1. Persistent logging and atomic saves

Nothing important depends on notebook output cells remaining intact.

The controller writes:

- `overnight.log`
- `events.jsonl`
- `training_metrics.jsonl`
- `snapshot_metrics.jsonl`
- `errors.jsonl`
- `run_state.json`
- a few model-only checkpoints

as the run proceeds.


In [ ]:
RUN_DIR: Path | None = None
LOG_PATH: Path | None = None
EVENTS_PATH: Path | None = None
TRAIN_METRICS_PATH: Path | None = None
SNAPSHOT_METRICS_PATH: Path | None = None
ERRORS_PATH: Path | None = None
STATE_PATH: Path | None = None

RUN_CLOCK: dict[str, float] = {}


def now_text() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")


def _jsonable(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if torch.is_tensor(value):
        if value.numel() == 1:
            return value.detach().cpu().item()
        return value.detach().cpu().tolist()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def log(message: str) -> None:
    line = f"[{now_text()}] {message}"
    print(line, flush=True)
    if LOG_PATH is not None:
        with LOG_PATH.open("a", encoding="utf-8") as handle:
            handle.write(line + "\n")
            handle.flush()


def append_jsonl(path: Path | None, payload: dict[str, Any]) -> None:
    if path is None:
        return
    record = {
        key: _jsonable(value)
        for key, value in payload.items()
    }
    with path.open("a", encoding="utf-8") as handle:
        json.dump(record, handle, ensure_ascii=False)
        handle.write("\n")
        handle.flush()


def atomic_json(path: Path, payload: dict[str, Any]) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as handle:
        json.dump(
            {k: _jsonable(v) for k, v in payload.items()},
            handle,
            indent=2,
            ensure_ascii=False,
        )
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp, path)


def record_event(kind: str, **payload: Any) -> None:
    append_jsonl(
        EVENTS_PATH,
        {
            "time": now_text(),
            "kind": kind,
            **payload,
        },
    )


def record_error(stage: str, exc: BaseException) -> None:
    text = "".join(
        traceback.format_exception(
            type(exc),
            exc,
            exc.__traceback__,
        )
    )
    log(f"ERROR in {stage}: {type(exc).__name__}: {exc}")
    append_jsonl(
        ERRORS_PATH,
        {
            "time": now_text(),
            "stage": stage,
            "exception_type": type(exc).__name__,
            "message": str(exc),
            "traceback": text,
        },
    )


def update_state(**kwargs: Any) -> None:
    if STATE_PATH is None:
        return
    previous = {}
    if STATE_PATH.exists():
        try:
            previous = json.loads(
                STATE_PATH.read_text(encoding="utf-8")
            )
        except Exception:
            previous = {}
    previous.update(
        {
            "updated_at": now_text(),
            **kwargs,
        }
    )
    atomic_json(STATE_PATH, previous)


def free_disk_gib(path: Path) -> float:
    root = Path(path.anchor) if path.anchor else path
    return shutil.disk_usage(root).free / 1024**3


def elapsed_hours() -> float:
    start = RUN_CLOCK.get("start", time.time())
    return (time.time() - start) / 3600.0


def training_time_left_seconds() -> float:
    return max(
        0.0,
        RUN_CLOCK.get("training_deadline", time.time())
        - time.time(),
    )


def total_time_left_seconds() -> float:
    return max(
        0.0,
        RUN_CLOCK.get("hard_deadline", time.time())
        - time.time(),
    )


def within_stage_deadline(deadline: float) -> bool:
    return (
        time.time() < deadline
        and training_time_left_seconds() > 0
    )


def safe_cuda_cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def atomic_model_checkpoint(
    path: Path,
    *,
    model: StirNet,
    cfg,
    step: int,
    extra: dict[str, Any],
) -> None:
    '''
    Model-only checkpoint. Optimizer/scaler states are deliberately omitted
    to keep disk use low. A temporary file is replaced atomically.
    '''
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    if tmp.exists():
        tmp.unlink()
    save_checkpoint(
        tmp,
        model=model,
        optimizer=None,
        scheduler=None,
        scaler=None,
        step=int(step),
        config=cfg,
        extra=extra,
    )
    os.replace(tmp, path)
    log(
        f"Saved {path.name} "
        f"({path.stat().st_size / 1024**2:.1f} MiB)"
    )


def checkpoint_file_size_gib() -> float:
    if RUN_DIR is None:
        return 0.0
    return sum(
        p.stat().st_size
        for p in RUN_DIR.glob("*.pt")
        if p.is_file()
    ) / 1024**3


## 2. Data and model helpers

These functions use the current repository's actual proposal/query interfaces.

The spatial-only path is the same structural path used in Notebook 24:
- no CR1/CR2,
- empty temporal state,
- current proposal generator and query decoder.

The joint stage later returns to the repository's normal full forward.


In [ ]:
batch_cpu = None
b = None
targets = None
target = None

current_labels_native = None
gt_labels_native = None
spacing_native = None
dref_um = None

gt_ids = None
gt_centers_cellscale = None
source9_gt_ids = None
source9_gt_id_set = None
source9_gt_indices = None
source9_gt_centers = None

GT_FOREGROUND = None
GT_INTERNAL = None
GT_ALL_BOUNDARY = None
SOURCE9_NATIVE = None


def make_empty_temporal(model: StirNet, *, dtype: torch.dtype) -> TemporalState:
    d_model = int(model.cfg.temporal.d_model)
    return TemporalState(
        tokens=torch.empty((0, d_model), device=device, dtype=dtype),
        ref_um=torch.empty((0, 3), device=device, dtype=torch.float32),
        ref_cellscale=torch.empty((0, 3), device=device, dtype=torch.float32),
        salience=torch.empty((0, 1), device=device, dtype=dtype),
        reliability=torch.empty((0, 1), device=device, dtype=dtype),
        status=torch.empty((0,), device=device, dtype=torch.long),
        edge_index=torch.empty((2, 0), device=device, dtype=torch.long),
        edge_attr=torch.empty((0, 22), device=device, dtype=torch.float32),
        batch_index=torch.empty((0,), device=device, dtype=torch.long),
    )


def new_cfg():
    cfg = _reduced_config()
    cfg.proposals.enabled = True
    cfg.proposals.query_mode = "spatial_proposals"
    return cfg


def load_fresh_model(checkpoint: Path = STEP30_CHECKPOINT):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    cfg = new_cfg()
    model = StirNet(cfg).to(device)

    info = load_checkpoint(
        checkpoint,
        model,
        optimizer=None,
        scheduler=None,
        scaler=None,
        map_location="cpu",
        strict=True,
        migrate_history=True,
    )
    return model, cfg, info


def forward_spatial_backbone(model: StirNet) -> dict[str, Any]:
    acq = model.acquisition(
        b["spacing_um"],
        b["dref_um"],
    )

    pyramid = model.encoder(
        b["spatial_inputs"],
        b["spacing_um"],
        acq,
        b.get("spatial_padding_mask"),
    )

    e3 = pyramid.features[3]

    # Strictly spatial-only: CR1/CR2 are not called.
    e2 = model.decoder.decode_to_e2(
        e3,
        pyramid,
        acq,
    )

    d1, d0, mask_features = model.decoder.decode_from_e2(
        e2,
        pyramid,
        acq,
    )

    dense = model.dense_heads(d0)

    proposal_state, proposal_score_logits = (
        model.spatial_proposal_generator(
            d0,
            e2,
            b["spatial_inputs"],
            dense,
            b["instance_labels"],
            b["spacing_um"],
            pyramid.spacings_um[2],
            b["dref_um"],
            b["instance_ids"],
            b["instance_batch"],
            b["instance_centroids_um"],
            b.get("spatial_padding_mask"),
        )
    )

    dense = dict(dense)
    dense["proposal_score_logits"] = proposal_score_logits

    return {
        "acq": acq,
        "pyramid": pyramid,
        "e3": e3,
        "e2": e2,
        "d1": d1,
        "d0": d0,
        "mask_features": mask_features,
        "dense": dense,
        "proposals": proposal_state,
    }


def source9_query_rows(qstate: QueryState) -> torch.Tensor:
    return torch.nonzero(
        (~qstate.padding_mask[0])
        & (qstate.query_types[0] == QUERY_SPATIAL_PROPOSAL)
        & (qstate.source_instance_ids[0] == SOURCE_ID),
        as_tuple=False,
    ).flatten()


def transform_qstate(
    qstate: QueryState,
    mode: str | None,
) -> QueryState:
    '''
    Causal anchor-vs-query ablations on source-9 spatial proposal queries.
    '''
    if mode is None or mode == "normal":
        return qstate

    rows = source9_query_rows(qstate)
    if rows.numel() < 2:
        return qstate

    embeddings = qstate.embeddings.clone()
    references = qstate.references_cellscale.clone()
    initial = (
        qstate.initial_references_cellscale.clone()
        if qstate.initial_references_cellscale is not None
        else references.clone()
    )

    if mode == "identical_query":
        mean_q = embeddings[0, rows].mean(dim=0, keepdim=True)
        embeddings[0, rows] = mean_q.expand(len(rows), -1)

    elif mode == "collapsed_anchor":
        mean_ref = references[0, rows].mean(dim=0, keepdim=True)
        references[0, rows] = mean_ref.expand(len(rows), -1)
        initial[0, rows] = mean_ref.expand(len(rows), -1)

    elif mode == "shuffled_query":
        # Deterministic reversal avoids introducing RNG variability.
        embeddings[0, rows] = embeddings[0, rows.flip(0)]

    else:
        raise ValueError(f"Unknown qstate ablation: {mode}")

    return replace(
        qstate,
        embeddings=embeddings,
        references_cellscale=references,
        initial_references_cellscale=initial,
    )


def forward_spatial_only_full(
    model: StirNet,
    *,
    qstate_ablation: str | None = None,
) -> StirNetOutput:
    spatial = forward_spatial_backbone(model)

    e3 = spatial["e3"]
    e2 = spatial["e2"]
    d1 = spatial["d1"]
    pyramid = spatial["pyramid"]
    dense = spatial["dense"]
    proposal_state = spatial["proposals"]

    temporal = make_empty_temporal(
        model,
        dtype=e2.dtype,
    )

    qstate = model.query_builder(
        e2,
        pyramid.spacings_um[2],
        b["instance_labels"],
        b["instance_features"],
        b["instance_ids"],
        b["instance_batch"],
        b["instance_centroids_um"],
        b["dref_um"],
        temporal,
        memory_ablation="full",
        return_debug=False,
        full_attention=False,
        proposal_state=proposal_state,
        query_mode="spatial_proposals",
    )

    qstate = transform_qstate(
        qstate,
        qstate_ablation,
    )

    initial_references = qstate.references_cellscale.clone()
    initial_embeddings = qstate.embeddings.clone()

    qstate, decoder_outputs = model.query_decoder(
        qstate,
        [e3, e2, d1],
        [
            pyramid.spacings_um[3],
            pyramid.spacings_um[2],
            pyramid.spacings_um[1],
        ],
        b["instance_labels"],
        b["dref_um"],
        temporal,
        memory_ablation="full",
        return_debug=False,
        full_attention=False,
    )

    final = decoder_outputs[-1]
    native_embeddings = model.native_mask_head(qstate.embeddings)

    return StirNetOutput(
        exist_logits=final["exist_logits"],
        centers_cellscale=final["centers_cellscale"],
        coarse_mask_logits=final["coarse_mask_logits"],
        coarse_spacing_um=final["coarse_spacing_um"],
        query_embeddings=qstate.embeddings,
        native_mask_embeddings=native_embeddings,
        query_types=qstate.query_types,
        query_padding_mask=qstate.padding_mask,
        source_instance_ids=qstate.source_instance_ids,
        query_initial_references_cellscale=initial_references,
        temporal_salience=qstate.temporal_salience,
        temporal_reliability=qstate.temporal_reliability,
        aux_outputs=decoder_outputs[:-1],
        dense_outputs=dense,
        mask_features=spatial["mask_features"],
        spacing_um=b["spacing_um"],
        dref_um=b["dref_um"],
        instance_labels=b["instance_labels"],
        debug={
            "initial_query_embeddings": initial_embeddings,
            "query_layer_references_cellscale": torch.stack(
                [
                    layer["centers_cellscale"]
                    for layer in decoder_outputs
                ],
                dim=0,
            ),
            "qstate_ablation": qstate_ablation or "normal",
        },
        proposals=proposal_state,
    )


@torch.no_grad()
def forward_full_model(model: StirNet) -> StirNetOutput:
    model.eval()
    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=device.type == "cuda",
    ):
        return model_forward_from_batch(
            model,
            b,
            bypass_coreasoning=False,
            return_debug=False,
        )


## 3. Metrics: proposal cardinality + support-aligned instance masks

Notebook 24's whole-volume coarse Dice was not aligned with the actual training support.

Notebook 27 therefore uses the repository's **same physical GT-local support rule** as `RefinementCriterion`.

The per-cell table reports:

- proposal distance / duplicate count;
- matched query;
- initial/final center;
- existence probability;
- local soft Dice;
- local hard Dice;
- local precision / recall / IoU;
- center error.

The main metric for the query/native stages is **source-9 GT-local mask Dice**, not whole-volume Dice.


In [ ]:
def _safe_float(x) -> float:
    if torch.is_tensor(x):
        x = x.detach().float().cpu().item()
    return float(x)


def pairwise_soft_dice(logits: torch.Tensor) -> float:
    if logits.shape[0] < 2:
        return float("nan")
    p = logits.float().sigmoid().flatten(1)
    inter = p @ p.T
    sums = p.sum(1)
    dice = (2 * inter + 1e-6) / (
        sums[:, None] + sums[None, :] + 1e-6
    )
    tri = torch.triu(
        torch.ones_like(dice, dtype=torch.bool),
        diagonal=1,
    )
    return float(dice[tri].mean().detach().cpu())


def proposal_statistics(proposals) -> tuple[dict[str, Any], pd.DataFrame]:
    valid = ~proposals.padding_mask[0].detach().cpu()
    refs = (
        proposals.references_cellscale[0]
        .detach()
        .float()
        .cpu()[valid]
    )
    scores = (
        proposals.scores[0]
        .detach()
        .float()
        .cpu()[valid]
    )
    sources = (
        proposals.source_instance_ids[0]
        .detach()
        .cpu()[valid]
    )
    fallback = (
        proposals.fallback_mask[0]
        .detach()
        .cpu()[valid]
    )

    learned = ~fallback
    learned_refs = refs[learned]

    all_gt_centers = gt_centers_cellscale.float().cpu()

    if len(learned_refs):
        all_dist = torch.cdist(
            all_gt_centers,
            learned_refs,
        )
        all_nearest = all_dist.min(dim=1).values
    else:
        all_dist = torch.empty(
            (len(all_gt_centers), 0),
            dtype=torch.float32,
        )
        all_nearest = torch.full(
            (len(all_gt_centers),),
            float("inf"),
        )

    if len(learned_refs):
        s9_dist = torch.cdist(
            source9_gt_centers.float().cpu(),
            learned_refs,
        )
        s9_nearest = s9_dist.min(dim=1).values
        s9_dup_05 = (s9_dist <= 0.5).sum(dim=1)
        s9_dup_10 = (s9_dist <= 1.0).sum(dim=1)
    else:
        s9_nearest = torch.full(
            (len(source9_gt_centers),),
            float("inf"),
        )
        s9_dup_05 = torch.zeros(
            len(source9_gt_centers),
            dtype=torch.long,
        )
        s9_dup_10 = s9_dup_05.clone()

    source9_associated = (
        (sources == SOURCE_ID) & learned
    )

    row = {
        "proposal_total": int(valid.sum()),
        "proposal_learned": int(learned.sum()),
        "proposal_fallback": int(fallback.sum()),
        "proposal_off_mask_learned": int(
            ((sources < 0) & learned).sum()
        ),
        "source9_associated_learned": int(
            source9_associated.sum()
        ),
        "source9_fallback": int(
            ((sources == SOURCE_ID) & fallback).sum()
        ),
        "all_gt_recall_0p5": float(
            (all_nearest <= 0.5).float().mean()
        ),
        "all_gt_recall_1p0": float(
            (all_nearest <= 1.0).float().mean()
        ),
        "source9_recall_0p5": float(
            (s9_nearest <= 0.5).float().mean()
        ),
        "source9_recall_1p0": float(
            (s9_nearest <= 1.0).float().mean()
        ),
        "source9_nearest_mean_dref": float(
            s9_nearest.mean()
        ),
        "source9_nearest_max_dref": float(
            s9_nearest.max()
        ),
        "source9_mean_proposals_within_0p5": float(
            s9_dup_05.float().mean()
        ),
        "source9_max_proposals_within_0p5": int(
            s9_dup_05.max()
        ),
        "source9_gt_with_duplicate_0p5": int(
            (s9_dup_05 > 1).sum()
        ),
        "source9_mean_proposals_within_1p0": float(
            s9_dup_10.float().mean()
        ),
    }

    per_gt = pd.DataFrame(
        {
            "gt_id": source9_gt_ids,
            "nearest_proposal_dref": s9_nearest.numpy(),
            "proposals_within_0p5": s9_dup_05.numpy(),
            "proposals_within_1p0": s9_dup_10.numpy(),
        }
    )

    return row, per_gt


def support_aligned_source9_metrics(
    outputs: StirNetOutput,
) -> tuple[dict[str, Any], pd.DataFrame, Any]:
    matching = run_matching_probe(
        outputs,
        targets,
    )
    match = matching.matches[0]

    qtypes = outputs.query_types[0].detach().cpu()
    sources = outputs.source_instance_ids[0].detach().cpu()

    shape = tuple(
        int(v)
        for v in outputs.coarse_mask_logits.shape[-3:]
    )

    source9_indices_cpu = source9_gt_indices.detach().cpu().long()

    s9_masks = target_masks_at_shape(
        target,
        shape,
        outputs.coarse_mask_logits.device,
        target_indices=source9_indices_cpu,
    )

    s9_centers = gt_centers_cellscale[
        source9_indices_cpu
    ].to(
        outputs.coarse_mask_logits.device,
        dtype=torch.float32,
    )

    spacing = outputs.coarse_spacing_um
    spacing_b = spacing[0] if spacing.ndim == 2 else spacing

    dref = outputs.dref_um
    dref_b = dref[0] if dref.ndim else dref

    support = build_local_support_masks(
        s9_masks,
        s9_centers,
        spacing_b,
        dref_b,
        float(outputs.coarse_mask_logits.new_tensor(
            new_cfg().losses.mask_supervision_radius_dref
        ).item()),
    )

    gt_index_to_s9_row = {
        int(global_idx): row
        for row, global_idx in enumerate(
            source9_indices_cpu.tolist()
        )
    }

    records = []
    matched_source9_proposal_queries = []

    for pred_idx, target_idx in zip(
        match.pred_indices.detach().cpu().tolist(),
        match.target_indices.detach().cpu().tolist(),
    ):
        pred_idx = int(pred_idx)
        target_idx = int(target_idx)

        if target_idx not in gt_index_to_s9_row:
            continue

        gt_row = gt_index_to_s9_row[target_idx]
        gt_id = int(gt_ids[target_idx])

        pred = (
            outputs.coarse_mask_logits[0, pred_idx]
            .float()
        )
        probability = pred.sigmoid()
        target_mask = s9_masks[gt_row].bool()
        local = support[gt_row].bool()

        p_local = probability * local.float()
        target_float = target_mask.float()

        intersection = (
            probability * target_float
        ).sum()

        soft_dice = (
            2 * intersection + 1e-6
        ) / (
            p_local.sum()
            + target_float.sum()
            + 1e-6
        )

        hard = (
            (probability >= 0.5)
            & local
        )
        tp = (hard & target_mask).sum().float()
        fp = (hard & ~target_mask).sum().float()
        fn = (~hard & target_mask).sum().float()

        hard_dice = (
            2 * tp + 1e-6
        ) / (
            2 * tp + fp + fn + 1e-6
        )
        precision = (
            tp + 1e-6
        ) / (
            tp + fp + 1e-6
        )
        recall = (
            tp + 1e-6
        ) / (
            tp + fn + 1e-6
        )
        iou = (
            tp + 1e-6
        ) / (
            tp + fp + fn + 1e-6
        )

        initial_ref = (
            outputs.query_initial_references_cellscale[
                0, pred_idx
            ]
            .detach()
            .float()
            .cpu()
        )
        final_ref = (
            outputs.centers_cellscale[
                0, pred_idx
            ]
            .detach()
            .float()
            .cpu()
        )
        gt_center = (
            gt_centers_cellscale[target_idx]
            .detach()
            .float()
            .cpu()
        )

        center_error_dref = float(
            torch.linalg.vector_norm(
                final_ref - gt_center
            )
        )

        qtype = int(qtypes[pred_idx])
        source_id = int(sources[pred_idx])

        if qtype == QUERY_SPATIAL_PROPOSAL:
            matched_source9_proposal_queries.append(
                pred_idx
            )

        records.append(
            {
                "gt_id": gt_id,
                "query_index": pred_idx,
                "query_type": qtype,
                "source_id": source_id,
                "exist_prob": float(
                    outputs.exist_logits[
                        0, pred_idx
                    ]
                    .detach()
                    .float()
                    .sigmoid()
                    .cpu()
                ),
                "initial_z_dref": float(initial_ref[0]),
                "initial_y_dref": float(initial_ref[1]),
                "initial_x_dref": float(initial_ref[2]),
                "final_z_dref": float(final_ref[0]),
                "final_y_dref": float(final_ref[1]),
                "final_x_dref": float(final_ref[2]),
                "center_error_dref": center_error_dref,
                "center_error_um": center_error_dref * dref_um,
                "local_soft_dice": float(
                    soft_dice.detach().cpu()
                ),
                "local_hard_dice": float(
                    hard_dice.detach().cpu()
                ),
                "local_precision": float(
                    precision.detach().cpu()
                ),
                "local_recall": float(
                    recall.detach().cpu()
                ),
                "local_iou": float(
                    iou.detach().cpu()
                ),
                "predicted_local_voxels": int(
                    hard.sum().detach().cpu()
                ),
                "gt_voxels_coarse": int(
                    target_mask.sum().detach().cpu()
                ),
            }
        )

    per_cell = pd.DataFrame(records)

    if len(per_cell):
        proposal_only = per_cell[
            per_cell["query_type"]
            == QUERY_SPATIAL_PROPOSAL
        ]
    else:
        proposal_only = per_cell

    source9_query_rows_all = torch.nonzero(
        (~outputs.query_padding_mask[0].detach().cpu())
        & (
            outputs.query_types[0].detach().cpu()
            == QUERY_SPATIAL_PROPOSAL
        )
        & (
            outputs.source_instance_ids[0].detach().cpu()
            == SOURCE_ID
        ),
        as_tuple=False,
    ).flatten()

    source9_logits = (
        outputs.coarse_mask_logits[
            0,
            source9_query_rows_all.to(
                outputs.coarse_mask_logits.device
            ),
        ]
        if len(source9_query_rows_all)
        else outputs.coarse_mask_logits.new_zeros(
            (0, *outputs.coarse_mask_logits.shape[-3:])
        )
    )

    summary = {
        "source9_gt_matched_any_query": int(
            per_cell["gt_id"].nunique()
        ) if len(per_cell) else 0,
        "source9_gt_matched_proposal_query": int(
            proposal_only["gt_id"].nunique()
        ) if len(proposal_only) else 0,
        "source9_local_soft_dice_mean": float(
            proposal_only["local_soft_dice"].mean()
        ) if len(proposal_only) else float("nan"),
        "source9_local_soft_dice_min": float(
            proposal_only["local_soft_dice"].min()
        ) if len(proposal_only) else float("nan"),
        "source9_local_hard_dice_mean": float(
            proposal_only["local_hard_dice"].mean()
        ) if len(proposal_only) else float("nan"),
        "source9_local_precision_mean": float(
            proposal_only["local_precision"].mean()
        ) if len(proposal_only) else float("nan"),
        "source9_local_recall_mean": float(
            proposal_only["local_recall"].mean()
        ) if len(proposal_only) else float("nan"),
        "source9_local_iou_mean": float(
            proposal_only["local_iou"].mean()
        ) if len(proposal_only) else float("nan"),
        "source9_center_error_um_mean": float(
            proposal_only["center_error_um"].mean()
        ) if len(proposal_only) else float("nan"),
        "source9_seed_query_count": int(
            len(source9_query_rows_all)
        ),
        "source9_pairwise_coarse_soft_dice": (
            pairwise_soft_dice(source9_logits)
            if len(source9_query_rows_all) >= 2
            else float("nan")
        ),
    }

    return summary, per_cell, matching


def dense_metrics(outputs: StirNetOutput) -> dict[str, Any]:
    fg_pred = (
        outputs.dense_outputs["foreground_logits"][0, 0]
        .detach()
        .float()
        .cpu()
        .sigmoid()
        >= 0.5
    )

    intersection = (
        fg_pred & GT_FOREGROUND
    ).sum().float()

    foreground_dice = float(
        (
            2 * intersection + 1e-6
        ) / (
            fg_pred.sum()
            + GT_FOREGROUND.sum()
            + 1e-6
        )
    )

    return {
        "foreground_hard_dice": foreground_dice,
    }


@torch.no_grad()
def evaluate_outputs(
    outputs: StirNetOutput,
    *,
    label: str,
    stage: str,
    step: int,
) -> tuple[dict[str, Any], pd.DataFrame]:
    proposal_row, proposal_per_gt = proposal_statistics(
        outputs.proposals
    )
    mask_row, per_cell, _ = (
        support_aligned_source9_metrics(
            outputs
        )
    )
    dense_row = dense_metrics(outputs)

    row = {
        "time": now_text(),
        "label": label,
        "stage": stage,
        "step": int(step),
        "elapsed_hours": elapsed_hours(),
        **proposal_row,
        **mask_row,
        **dense_row,
        "free_disk_gib": free_disk_gib(REPO_ROOT),
        "checkpoint_disk_gib": checkpoint_file_size_gib(),
    }

    if len(per_cell):
        per_cell = per_cell.merge(
            proposal_per_gt,
            how="left",
            on="gt_id",
        )

    append_jsonl(
        SNAPSHOT_METRICS_PATH,
        row,
    )

    return row, per_cell


## 4. Compact native-resolution source-9 evaluator

This does **not** save full 4-D query probabilities.

It renders matched source-9 spatial-proposal queries one at a time inside a physically padded crop and immediately reduces them to metrics plus one combined label volume.


In [ ]:
SOURCE9_CROP_SLICES = None
SOURCE9_CROP_LO = None
SOURCE9_CROP_HI = None
SOURCE9_CROP_SHAPE = None


def setup_source9_crop(
    margin_dref: float = 3.0,
) -> None:
    global SOURCE9_CROP_SLICES
    global SOURCE9_CROP_LO
    global SOURCE9_CROP_HI
    global SOURCE9_CROP_SHAPE

    source_voxels = np.argwhere(
        current_labels_native == SOURCE_ID
    )
    if len(source_voxels) == 0:
        raise RuntimeError(
            "Source component 9 is empty."
        )

    lo = source_voxels.min(axis=0)
    hi = source_voxels.max(axis=0) + 1

    margin_um = margin_dref * dref_um
    margin_vox = np.ceil(
        margin_um / spacing_native
    ).astype(np.int64)

    full_shape = np.asarray(
        current_labels_native.shape,
        dtype=np.int64,
    )

    lo = np.maximum(0, lo - margin_vox)
    hi = np.minimum(full_shape, hi + margin_vox)

    SOURCE9_CROP_LO = lo
    SOURCE9_CROP_HI = hi
    SOURCE9_CROP_SHAPE = tuple(
        (hi - lo).tolist()
    )
    SOURCE9_CROP_SLICES = tuple(
        slice(int(a), int(z))
        for a, z in zip(lo, hi)
    )


def crop_coordinates_um() -> torch.Tensor:
    lo = SOURCE9_CROP_LO
    hi = SOURCE9_CROP_HI

    z = torch.arange(
        int(lo[0]),
        int(hi[0]),
        device=device,
        dtype=torch.float32,
    )
    y = torch.arange(
        int(lo[1]),
        int(hi[1]),
        device=device,
        dtype=torch.float32,
    )
    x = torch.arange(
        int(lo[2]),
        int(hi[2]),
        device=device,
        dtype=torch.float32,
    )

    zz, yy, xx = torch.meshgrid(
        z, y, x, indexing="ij"
    )
    coords_vox = torch.stack(
        [zz, yy, xx],
        dim=-1,
    ).reshape(-1, 3)

    spacing = torch.as_tensor(
        spacing_native,
        device=device,
        dtype=torch.float32,
    )

    full_shape = torch.as_tensor(
        np.asarray(current_labels_native.shape) - 1,
        device=device,
        dtype=torch.float32,
    )

    extent = full_shape * spacing

    return (
        coords_vox * spacing[None]
        - 0.5 * extent[None]
    )


@torch.no_grad()
def native_source9_metrics(
    outputs: StirNetOutput,
    *,
    save_compact_npz: Path | None = None,
) -> tuple[dict[str, Any], pd.DataFrame]:
    _, per_cell, matching = (
        support_aligned_source9_metrics(
            outputs
        )
    )

    if not len(per_cell):
        return {
            "native_source9_cells": 0,
            "native_soft_dice_mean": float("nan"),
            "native_hard_dice_mean": float("nan"),
        }, pd.DataFrame()

    proposal_cells = per_cell[
        per_cell["query_type"]
        == QUERY_SPATIAL_PROPOSAL
    ].copy()

    if not len(proposal_cells):
        return {
            "native_source9_cells": 0,
            "native_soft_dice_mean": float("nan"),
            "native_hard_dice_mean": float("nan"),
        }, pd.DataFrame()

    coords_um = crop_coordinates_um()

    crop_features = outputs.mask_features[
        0,
        :,
        SOURCE9_CROP_SLICES[0],
        SOURCE9_CROP_SLICES[1],
        SOURCE9_CROP_SLICES[2],
    ].float()

    current_crop_flat = b["instance_labels"][
        0,
        SOURCE9_CROP_SLICES[0],
        SOURCE9_CROP_SLICES[1],
        SOURCE9_CROP_SLICES[2],
    ].flatten()

    gt_crop = gt_labels_native[
        SOURCE9_CROP_SLICES
    ]

    best_probability = np.zeros(
        SOURCE9_CROP_SHAPE,
        dtype=np.float16,
    )
    best_local_id = np.zeros(
        SOURCE9_CROP_SHAPE,
        dtype=np.int16,
    )

    records = []

    for local_id, record in enumerate(
        proposal_cells.to_dict("records"),
        start=1,
    ):
        query_idx = int(record["query_index"])
        gt_id = int(record["gt_id"])

        embedding = (
            outputs.native_mask_embeddings[
                0, query_idx
            ]
            .float()
            .unsqueeze(0)
        )

        learned_logits = torch.einsum(
            "qc,cv->qv",
            embedding,
            crop_features.flatten(1),
        )

        qtype = outputs.query_types[
            0, query_idx
        ].reshape(1)

        source_id = outputs.source_instance_ids[
            0, query_idx
        ].reshape(1)

        ref_um = (
            outputs.centers_cellscale[
                0, query_idx
            ]
            .float()
            .reshape(1, 3)
            * outputs.dref_um[0].float()
        )

        # Spatial-proposal native support does not use the source component.
        source_support = torch.zeros(
            (1, coords_um.shape[0]),
            device=device,
            dtype=torch.bool,
        )

        logits, _, _ = compose_native_query_logits(
            learned_logits,
            qtype,
            source_id,
            ref_um,
            current_crop_flat,
            source_support,
            coords_um,
            outputs.dref_um[0],
            support_radius_dref=(
                new_cfg().queries.native_support_radius_dref
            ),
            temporal_sigma_dref=(
                new_cfg().queries.temporal_gaussian_sigma_dref
            ),
            prior_inside_logit=(
                new_cfg().queries.prior_inside_logit
            ),
            prior_outside_logit=(
                new_cfg().queries.prior_outside_logit
            ),
            background_logit=(
                new_cfg().queries.native_background_logit
            ),
            proposal_support_radius_dref=(
                new_cfg().proposals.native_support_radius_dref
            ),
        )

        probability = (
            logits[0]
            .reshape(SOURCE9_CROP_SHAPE)
            .sigmoid()
        )

        target_mask_np = (
            gt_crop == gt_id
        )
        target_mask = torch.from_numpy(
            target_mask_np
        ).to(
            device=device,
            dtype=torch.bool,
        )

        # The produced proposal mask is already radially support-bounded.
        p = probability.float()
        target_float = target_mask.float()

        intersection = (
            p * target_float
        ).sum()

        soft_dice = (
            2 * intersection + 1e-6
        ) / (
            p.sum()
            + target_float.sum()
            + 1e-6
        )

        hard = p >= 0.5
        tp = (hard & target_mask).sum().float()
        fp = (hard & ~target_mask).sum().float()
        fn = (~hard & target_mask).sum().float()

        hard_dice = (
            2 * tp + 1e-6
        ) / (
            2 * tp + fp + fn + 1e-6
        )

        prob_np = (
            p.detach()
            .to("cpu", dtype=torch.float16)
            .numpy()
        )

        replace_mask = (
            prob_np > best_probability
        )
        best_probability[replace_mask] = (
            prob_np[replace_mask]
        )
        best_local_id[replace_mask] = local_id

        records.append(
            {
                "gt_id": gt_id,
                "query_index": query_idx,
                "native_soft_dice": float(
                    soft_dice.detach().cpu()
                ),
                "native_hard_dice": float(
                    hard_dice.detach().cpu()
                ),
                "native_max_probability": float(
                    p.max().detach().cpu()
                ),
                "native_predicted_voxels_0p5": int(
                    hard.sum().detach().cpu()
                ),
                "native_gt_voxels": int(
                    target_mask.sum().detach().cpu()
                ),
            }
        )

        del (
            embedding,
            learned_logits,
            logits,
            probability,
            p,
            target_mask,
        )
        safe_cuda_cleanup()

    native_df = pd.DataFrame(records)

    combined_labels = np.where(
        best_probability >= np.float16(0.5),
        best_local_id,
        0,
    ).astype(np.int16)

    if save_compact_npz is not None:
        raw_crop = (
            batch_cpu["spatial_inputs"][
                0, 0
            ]
            .detach()
            .cpu()
            .numpy()[SOURCE9_CROP_SLICES]
            .astype(np.float16, copy=False)
        )

        np.savez_compressed(
            save_compact_npz,
            raw=raw_crop,
            current_labels=(
                current_labels_native[
                    SOURCE9_CROP_SLICES
                ]
                .astype(np.int16, copy=False)
            ),
            gt_labels=(
                gt_crop.astype(
                    np.int16,
                    copy=False,
                )
            ),
            best_probability=best_probability,
            predicted_labels=combined_labels,
            crop_lo=SOURCE9_CROP_LO,
            crop_hi=SOURCE9_CROP_HI,
            spacing_um=spacing_native,
            source9_gt_ids=source9_gt_ids,
        )

    summary = {
        "native_source9_cells": int(
            native_df["gt_id"].nunique()
        ),
        "native_soft_dice_mean": float(
            native_df["native_soft_dice"].mean()
        ),
        "native_soft_dice_min": float(
            native_df["native_soft_dice"].min()
        ),
        "native_hard_dice_mean": float(
            native_df["native_hard_dice"].mean()
        ),
    }

    return summary, native_df


## 5. Training utilities

The three spatial-only stages use fresh optimizers so their intended LR scales are explicit.

That is intentional: the starting checkpoint predates the proposal parameter group, so its old optimizer state cannot be safely migrated.

The full joint stage later uses the repository's `Trainer` and standard curriculum semantics.


In [ ]:
def configure_optimizer(
    model: StirNet,
    cfg,
    lr_scales: dict[str, float],
):
    groups = model_parameter_groups(model)

    for parameter in model.parameters():
        parameter.requires_grad_(False)

    params = []

    for group_name, scale in lr_scales.items():
        if group_name not in groups:
            raise KeyError(
                f"Unknown model parameter group: {group_name}"
            )
        for parameter in groups[group_name]:
            parameter.requires_grad_(True)
        params.append(
            {
                "name": group_name,
                "params": groups[group_name],
                "lr": BASE_LR * float(scale),
            }
        )

    optimizer = torch.optim.AdamW(
        params,
        lr=BASE_LR,
        weight_decay=cfg.training.weight_decay,
    )

    return optimizer


def new_scaler():
    return torch.amp.GradScaler(
        "cuda",
        enabled=device.type == "cuda",
        init_scale=1024.0,
    )


def proposal_criterion_for(cfg):
    return RefinementCriterion(
        cfg.losses,
        cfg.queries,
        cfg.training,
        cfg.proposals,
    ).to(device)


def proposal_spatial_objective(
    spatial: dict[str, Any],
    criterion: RefinementCriterion,
    cfg,
) -> dict[str, torch.Tensor]:
    dense = spatial["dense"]

    foreground, center, boundary = (
        criterion._dense_losses(
            dense,
            targets,
        )
    )

    internal = criterion._internal_boundary_loss(
        dense["boundary_logits"],
        targets,
    )

    proposal_center = criterion._stream_dense_loss(
        dense["proposal_score_logits"],
        targets,
        "center_heatmap",
        focal=True,
    )

    w = cfg.losses

    total = (
        float(w.foreground) * foreground
        + float(w.center_heatmap) * center
        + float(w.boundary) * boundary
        + float(w.internal_boundary) * internal
        + float(w.proposal_center) * proposal_center
    )

    return {
        "loss": total,
        "foreground": foreground,
        "center_heatmap": center,
        "boundary": boundary,
        "internal_boundary": internal,
        "proposal_center": proposal_center,
    }


def proposal_rank(row: dict[str, Any]) -> tuple:
    '''
    Lexicographic:
    1) retain tight source-9 recall,
    2) retain loose recall,
    3) approach one proposal per source-9 cell,
    4) reduce duplicate-near-GT count.
    '''
    return (
        round(float(row["source9_recall_0p5"]), 6),
        round(float(row["source9_recall_1p0"]), 6),
        -abs(
            int(row["source9_associated_learned"])
            - len(source9_gt_ids)
        ),
        -int(row["source9_gt_with_duplicate_0p5"]),
        -int(row["source9_fallback"]),
    )


def mask_rank(row: dict[str, Any]) -> tuple:
    dice = row.get(
        "source9_local_soft_dice_mean",
        float("nan"),
    )
    if dice is None or not math.isfinite(float(dice)):
        dice = -1.0

    return (
        int(
            row.get(
                "source9_gt_matched_proposal_query",
                0,
            )
        ),
        float(dice),
        float(
            row.get(
                "source9_recall_0p5",
                0.0,
            )
        ),
    )


def scalar_losses(losses: dict[str, torch.Tensor]) -> dict[str, float]:
    return {
        key: float(
            value.detach().float().cpu()
        )
        for key, value in losses.items()
        if torch.is_tensor(value)
        and value.numel() == 1
    }


def train_backward_step(
    *,
    model: StirNet,
    optimizer,
    scaler,
    loss: torch.Tensor,
    cfg,
) -> float:
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)

    grad_norm = torch.nn.utils.clip_grad_norm_(
        [
            p
            for p in model.parameters()
            if p.requires_grad
        ],
        float(cfg.training.max_grad_norm),
    )

    scaler.step(optimizer)
    scaler.update()

    return float(
        grad_norm.detach().float().cpu()
    )


def save_recovery(
    model: StirNet,
    cfg,
    stage: str,
    step: int,
) -> None:
    if RUN_DIR is None:
        return
    atomic_model_checkpoint(
        RUN_DIR / "checkpoint_recovery.pt",
        model=model,
        cfg=cfg,
        step=step,
        extra={
            "notebook": 27,
            "kind": "rolling_recovery",
            "stage": stage,
        },
    )
    update_state(
        recovery_stage=stage,
        recovery_step=int(step),
        recovery_checkpoint=str(
            RUN_DIR / "checkpoint_recovery.pt"
        ),
    )


## 6. Stage implementations

Each stage logs every optimization step immediately.

A failure in one stage raises back to the controller, which records the traceback and then moves on using the best surviving checkpoint.


In [ ]:
@torch.no_grad()
def evaluate_proposal_only(
    model: StirNet,
    *,
    label: str,
    step: int,
) -> tuple[dict[str, Any], pd.DataFrame]:
    """
    Proposal warm-up evaluation intentionally stops before QueryBuilder /
    QueryDecoder. A downstream query failure must not erase a useful proposal
    training night.
    """
    model.eval()
    torch.cuda.reset_peak_memory_stats()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=device.type == "cuda",
    ):
        spatial = forward_spatial_backbone(model)

    proposal_row, per_gt = proposal_statistics(
        spatial["proposals"]
    )

    fg_pred = (
        spatial["dense"]["foreground_logits"][0, 0]
        .detach()
        .float()
        .cpu()
        .sigmoid()
        >= 0.5
    )
    intersection = (fg_pred & GT_FOREGROUND).sum().float()
    foreground_dice = float(
        (2 * intersection + 1e-6)
        / (fg_pred.sum() + GT_FOREGROUND.sum() + 1e-6)
    )

    row = {
        "time": now_text(),
        "label": label,
        "stage": "proposal",
        "step": int(step),
        "elapsed_hours": elapsed_hours(),
        **proposal_row,
        "foreground_hard_dice": foreground_dice,
        "peak_cuda_gib": (
            torch.cuda.max_memory_allocated() / 1024**3
            if device.type == "cuda"
            else 0.0
        ),
        "free_disk_gib": free_disk_gib(REPO_ROOT),
        "checkpoint_disk_gib": checkpoint_file_size_gib(),
    }

    append_jsonl(SNAPSHOT_METRICS_PATH, row)

    del spatial
    safe_cuda_cleanup()
    return row, per_gt


def evaluate_spatial_model(
    model: StirNet,
    *,
    label: str,
    stage: str,
    step: int,
    ablation: str | None = None,
) -> tuple[dict[str, Any], pd.DataFrame]:
    model.eval()
    torch.cuda.reset_peak_memory_stats()

    with torch.no_grad(), torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=device.type == "cuda",
    ):
        outputs = forward_spatial_only_full(
            model,
            qstate_ablation=ablation,
        )

    row, per_cell = evaluate_outputs(
        outputs,
        label=label,
        stage=stage,
        step=step,
    )

    row["peak_cuda_gib"] = (
        torch.cuda.max_memory_allocated()
        / 1024**3
        if device.type == "cuda"
        else 0.0
    )

    del outputs
    safe_cuda_cleanup()

    return row, per_cell


def run_proposal_stage(
    model: StirNet,
    cfg,
    *,
    stage_deadline: float,
) -> dict[str, Any]:
    log("=== STAGE: proposal warm-up ===")

    criterion = proposal_criterion_for(cfg)

    optimizer = configure_optimizer(
        model,
        cfg,
        {
            "spatial": 1.0,
            "dense": 1.0,
            "proposal": 1.0,
        },
    )
    scaler = new_scaler()

    best_rank = None
    best_row = None
    best_path = RUN_DIR / "checkpoint_best_proposal.pt"

    for step in range(PROPOSAL_MAX_STEPS + 1):
        if (
            step == 0
            or step % PROPOSAL_EVAL_EVERY == 0
            or step == PROPOSAL_MAX_STEPS
        ):
            row, per_gt = evaluate_proposal_only(
                model,
                label=f"proposal_step_{step}",
                step=step,
            )

            rank = proposal_rank(row)

            log(
                "proposal eval "
                f"step={step} "
                f"source9={row['source9_associated_learned']} "
                f"recall0.5={row['source9_recall_0p5']:.3f} "
                f"recall1.0={row['source9_recall_1p0']:.3f} "
                f"dupGT={row['source9_gt_with_duplicate_0p5']}"
            )

            if best_rank is None or rank > best_rank:
                best_rank = rank
                best_row = row
                atomic_model_checkpoint(
                    best_path,
                    model=model,
                    cfg=cfg,
                    step=step,
                    extra={
                        "notebook": 27,
                        "stage": "proposal",
                        "rank": list(rank),
                    },
                )
                per_gt.to_csv(
                    RUN_DIR
                    / "best_proposal_per_gt.csv",
                    index=False,
                )
                update_state(
                    best_proposal_checkpoint=str(
                        best_path
                    ),
                    best_proposal_step=int(step),
                    best_proposal_rank=list(rank),
                )

        if step >= PROPOSAL_MAX_STEPS:
            break

        if not within_stage_deadline(
            stage_deadline
        ):
            log(
                "Proposal stage stopped by wall-clock deadline."
            )
            break

        model.train()
        criterion.train()
        optimizer.zero_grad(set_to_none=True)

        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
            enabled=device.type == "cuda",
        ):
            spatial = forward_spatial_backbone(
                model
            )
            losses = proposal_spatial_objective(
                spatial,
                criterion,
                cfg,
            )

        grad_norm = train_backward_step(
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            loss=losses["loss"],
            cfg=cfg,
        )

        metrics = {
            "time": now_text(),
            "stage": "proposal",
            "step": step + 1,
            "seconds": time.perf_counter() - t0,
            "grad_norm": grad_norm,
            "peak_cuda_gib": (
                torch.cuda.max_memory_allocated()
                / 1024**3
            ),
            **scalar_losses(losses),
        }
        append_jsonl(
            TRAIN_METRICS_PATH,
            metrics,
        )

        if (
            (step + 1)
            % RECOVERY_SAVE_EVERY
            == 0
        ):
            save_recovery(
                model,
                cfg,
                "proposal",
                step + 1,
            )

        del spatial, losses
        safe_cuda_cleanup()

    if best_path.exists():
        load_checkpoint(
            best_path,
            model,
            map_location="cpu",
            strict=True,
            migrate_history=True,
        )
        log(
            f"Restored best proposal checkpoint: {best_path.name}"
        )

    return {
        "best_path": best_path
        if best_path.exists()
        else None,
        "best_row": best_row,
    }


def run_anchor_query_ablation(
    model: StirNet,
) -> pd.DataFrame:
    log("=== EXPERIMENT: anchor-vs-query causal ablation ===")

    rows = []

    for mode in (
        "normal",
        "identical_query",
        "collapsed_anchor",
        "shuffled_query",
    ):
        try:
            row, per_cell = evaluate_spatial_model(
                model,
                label=f"ablation_{mode}",
                stage="anchor_query_ablation",
                step=0,
                ablation=mode,
            )
            row["ablation"] = mode
            rows.append(row)

            per_cell.to_csv(
                RUN_DIR
                / f"ablation_{mode}_per_cell.csv",
                index=False,
            )

            log(
                f"ablation={mode:16s} "
                f"matched={row['source9_gt_matched_proposal_query']} "
                f"localDice={row['source9_local_soft_dice_mean']}"
            )
        except Exception as exc:
            record_error(
                f"anchor_query_ablation/{mode}",
                exc,
            )

    df = pd.DataFrame(rows)
    if len(df):
        df.to_csv(
            RUN_DIR / "anchor_query_ablation.csv",
            index=False,
        )

    return df


def run_spatial_query_stage(
    model: StirNet,
    cfg,
    *,
    stage_deadline: float,
) -> dict[str, Any]:
    log("=== STAGE: spatial-only query training ===")

    optimizer = configure_optimizer(
        model,
        cfg,
        {
            "spatial": 0.25,
            "dense": 0.50,
            "proposal": 1.00,
            "query": 1.00,
        },
    )
    scaler = new_scaler()

    criterion = RefinementCriterion(
        cfg.losses,
        cfg.queries,
        cfg.training,
        cfg.proposals,
    ).to(device)

    criterion.set_loss_weight_overrides(
        {
            "dice_hi": 0.0,
            "focal_hi": 0.0,
            "count": 0.0,
            "overlap": 0.0,
        }
    )

    best_rank = None
    best_row = None
    best_path = RUN_DIR / "checkpoint_best_spatial_query.pt"

    for step in range(
        SPATIAL_QUERY_MAX_STEPS + 1
    ):
        if (
            step == 0
            or step % QUERY_EVAL_EVERY == 0
            or step == SPATIAL_QUERY_MAX_STEPS
        ):
            row, per_cell = evaluate_spatial_model(
                model,
                label=f"spatial_query_step_{step}",
                stage="spatial_query",
                step=step,
            )

            rank = mask_rank(row)

            log(
                "query eval "
                f"step={step} "
                f"matched={row['source9_gt_matched_proposal_query']}/9 "
                f"localDice={row['source9_local_soft_dice_mean']} "
                f"pairDice={row['source9_pairwise_coarse_soft_dice']}"
            )

            if best_rank is None or rank > best_rank:
                best_rank = rank
                best_row = row
                atomic_model_checkpoint(
                    best_path,
                    model=model,
                    cfg=cfg,
                    step=step,
                    extra={
                        "notebook": 27,
                        "stage": "spatial_query",
                        "rank": list(rank),
                    },
                )
                per_cell.to_csv(
                    RUN_DIR
                    / "best_spatial_query_per_cell.csv",
                    index=False,
                )
                update_state(
                    best_spatial_query_checkpoint=str(
                        best_path
                    ),
                    best_spatial_query_step=int(step),
                    best_spatial_query_rank=list(rank),
                )

        if step >= SPATIAL_QUERY_MAX_STEPS:
            break

        if not within_stage_deadline(
            stage_deadline
        ):
            log(
                "Spatial-query stage stopped by wall-clock deadline."
            )
            break

        model.train()
        criterion.train()
        optimizer.zero_grad(set_to_none=True)

        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
            enabled=device.type == "cuda",
        ):
            outputs = forward_spatial_only_full(
                model
            )
            losses = criterion(
                outputs,
                targets,
            )

        grad_norm = train_backward_step(
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            loss=losses["loss"],
            cfg=cfg,
        )

        append_jsonl(
            TRAIN_METRICS_PATH,
            {
                "time": now_text(),
                "stage": "spatial_query",
                "step": step + 1,
                "seconds": time.perf_counter() - t0,
                "grad_norm": grad_norm,
                "peak_cuda_gib": (
                    torch.cuda.max_memory_allocated()
                    / 1024**3
                ),
                **scalar_losses(losses),
            },
        )

        if (
            (step + 1)
            % RECOVERY_SAVE_EVERY
            == 0
        ):
            save_recovery(
                model,
                cfg,
                "spatial_query",
                step + 1,
            )

        del outputs, losses
        safe_cuda_cleanup()

    if best_path.exists():
        load_checkpoint(
            best_path,
            model,
            map_location="cpu",
            strict=True,
            migrate_history=True,
        )
        log(
            "Restored best spatial-query checkpoint."
        )

    return {
        "best_path": best_path
        if best_path.exists()
        else None,
        "best_row": best_row,
    }


def run_spatial_native_stage(
    model: StirNet,
    cfg,
    *,
    stage_deadline: float,
) -> dict[str, Any]:
    log("=== STAGE: spatial-only native-mask training ===")

    optimizer = configure_optimizer(
        model,
        cfg,
        {
            "spatial": 0.10,
            "dense": 0.50,
            "proposal": 0.50,
            "query": 1.00,
            "native": 1.00,
        },
    )
    scaler = new_scaler()

    criterion = RefinementCriterion(
        cfg.losses,
        cfg.queries,
        cfg.training,
        cfg.proposals,
    ).to(device)

    # Mirrors native_bootstrap: all normal losses except count/overlap.
    criterion.set_loss_weight_overrides(
        {
            "count": 0.0,
            "overlap": 0.0,
        }
    )

    best_rank = None
    best_row = None
    best_path = RUN_DIR / "checkpoint_best_native.pt"

    for step in range(
        SPATIAL_NATIVE_MAX_STEPS + 1
    ):
        if (
            step == 0
            or step % NATIVE_EVAL_EVERY == 0
            or step == SPATIAL_NATIVE_MAX_STEPS
        ):
            row, per_cell = evaluate_spatial_model(
                model,
                label=f"spatial_native_step_{step}",
                stage="spatial_native",
                step=step,
            )

            rank = mask_rank(row)

            log(
                "native-stage coarse eval "
                f"step={step} "
                f"matched={row['source9_gt_matched_proposal_query']}/9 "
                f"localDice={row['source9_local_soft_dice_mean']}"
            )

            if best_rank is None or rank > best_rank:
                best_rank = rank
                best_row = row
                atomic_model_checkpoint(
                    best_path,
                    model=model,
                    cfg=cfg,
                    step=step,
                    extra={
                        "notebook": 27,
                        "stage": "spatial_native",
                        "rank": list(rank),
                    },
                )
                per_cell.to_csv(
                    RUN_DIR
                    / "best_native_coarse_per_cell.csv",
                    index=False,
                )
                update_state(
                    best_native_checkpoint=str(
                        best_path
                    ),
                    best_native_step=int(step),
                    best_native_rank=list(rank),
                )

        if step >= SPATIAL_NATIVE_MAX_STEPS:
            break

        if not within_stage_deadline(
            stage_deadline
        ):
            log(
                "Spatial-native stage stopped by wall-clock deadline."
            )
            break

        model.train()
        criterion.train()
        optimizer.zero_grad(set_to_none=True)

        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
            enabled=device.type == "cuda",
        ):
            outputs = forward_spatial_only_full(
                model
            )
            losses = criterion(
                outputs,
                targets,
            )

        grad_norm = train_backward_step(
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            loss=losses["loss"],
            cfg=cfg,
        )

        append_jsonl(
            TRAIN_METRICS_PATH,
            {
                "time": now_text(),
                "stage": "spatial_native",
                "step": step + 1,
                "seconds": time.perf_counter() - t0,
                "grad_norm": grad_norm,
                "peak_cuda_gib": (
                    torch.cuda.max_memory_allocated()
                    / 1024**3
                ),
                **scalar_losses(losses),
            },
        )

        if (
            (step + 1)
            % RECOVERY_SAVE_EVERY
            == 0
        ):
            save_recovery(
                model,
                cfg,
                "spatial_native",
                step + 1,
            )

        del outputs, losses
        safe_cuda_cleanup()

    if best_path.exists():
        load_checkpoint(
            best_path,
            model,
            map_location="cpu",
            strict=True,
            migrate_history=True,
        )
        log(
            "Restored best spatial-native checkpoint."
        )

    return {
        "best_path": best_path
        if best_path.exists()
        else None,
        "best_row": best_row,
    }


def joint_start_step(cfg) -> int:
    c = cfg.curriculum
    return int(
        c.spatial_dense_steps
        + c.temporal_dense_steps
        + c.query_bootstrap_steps
        + c.native_bootstrap_steps
    )


def evaluate_joint_model(
    model: StirNet,
    *,
    label: str,
    step: int,
) -> tuple[dict[str, Any], pd.DataFrame]:
    model.eval()
    torch.cuda.reset_peak_memory_stats()

    with torch.no_grad(), torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=device.type == "cuda",
    ):
        outputs = model_forward_from_batch(
            model,
            b,
            bypass_coreasoning=False,
            return_debug=False,
        )

    row, per_cell = evaluate_outputs(
        outputs,
        label=label,
        stage="joint",
        step=step,
    )

    row["peak_cuda_gib"] = (
        torch.cuda.max_memory_allocated()
        / 1024**3
        if device.type == "cuda"
        else 0.0
    )

    del outputs
    safe_cuda_cleanup()

    return row, per_cell


def run_joint_stage(
    model: StirNet,
    cfg,
    *,
    stage_deadline: float,
) -> dict[str, Any]:
    log("=== STAGE: full-model joint training ===")

    trainer = Trainer(
        model,
        cfg,
        device=device,
        amp_dtype="fp16",
    )

    # The source checkpoint is step 30, i.e. immediately before the
    # repository's temporal_dense stage. Give CR/temporal modules a short,
    # bounded dense-only acclimation before asking the full model to optimize
    # instance masks jointly.
    trainer.global_step = int(
        cfg.curriculum.spatial_dense_steps
    )

    temporal_warmup_steps = 0
    while (
        temporal_warmup_steps < TEMPORAL_WARMUP_MAX_STEPS
        and within_stage_deadline(stage_deadline)
    ):
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        losses = trainer.train_step(b)
        temporal_warmup_steps += 1

        append_jsonl(
            TRAIN_METRICS_PATH,
            {
                "time": now_text(),
                "stage": "temporal_warmup",
                "step": temporal_warmup_steps,
                "global_step": trainer.global_step,
                "seconds": time.perf_counter() - t0,
                "peak_cuda_gib": (
                    torch.cuda.max_memory_allocated() / 1024**3
                ),
                **losses,
            },
        )

        log(
            "temporal warm-up "
            f"{temporal_warmup_steps}/{TEMPORAL_WARMUP_MAX_STEPS} "
            f"loss={losses.get('loss')}"
        )

    # Jump to the normal joint curriculum stage after the bounded temporal
    # acclimation. The next Trainer.train_step will re-apply the correct
    # trainability/LR scales without rebuilding the optimizer.
    trainer.global_step = joint_start_step(cfg)

    best_rank = None
    best_row = None
    best_path = RUN_DIR / "checkpoint_best_joint.pt"

    local_step = 0

    while (
        local_step <= JOINT_MAX_STEPS
        and within_stage_deadline(
            stage_deadline
        )
    ):
        if (
            local_step == 0
            or local_step % JOINT_EVAL_EVERY == 0
        ):
            row, per_cell = evaluate_joint_model(
                trainer.model,
                label=f"joint_step_{local_step}",
                step=local_step,
            )

            rank = mask_rank(row)

            log(
                "joint eval "
                f"step={local_step} "
                f"matched={row['source9_gt_matched_proposal_query']}/9 "
                f"localDice={row['source9_local_soft_dice_mean']}"
            )

            if best_rank is None or rank > best_rank:
                best_rank = rank
                best_row = row
                atomic_model_checkpoint(
                    best_path,
                    model=trainer.model,
                    cfg=cfg,
                    step=trainer.global_step,
                    extra={
                        "notebook": 27,
                        "stage": "joint",
                        "local_step": local_step,
                        "rank": list(rank),
                    },
                )
                per_cell.to_csv(
                    RUN_DIR
                    / "best_joint_per_cell.csv",
                    index=False,
                )
                update_state(
                    best_joint_checkpoint=str(
                        best_path
                    ),
                    best_joint_local_step=int(
                        local_step
                    ),
                    best_joint_rank=list(rank),
                )

        if local_step >= JOINT_MAX_STEPS:
            break

        if not within_stage_deadline(
            stage_deadline
        ):
            break

        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()

        losses = trainer.train_step(b)

        append_jsonl(
            TRAIN_METRICS_PATH,
            {
                "time": now_text(),
                "stage": "joint",
                "step": local_step + 1,
                "global_step": trainer.global_step,
                "seconds": time.perf_counter() - t0,
                "peak_cuda_gib": (
                    torch.cuda.max_memory_allocated()
                    / 1024**3
                ),
                **losses,
            },
        )

        local_step += 1

        if (
            local_step
            % RECOVERY_SAVE_EVERY
            == 0
        ):
            save_recovery(
                trainer.model,
                cfg,
                "joint",
                trainer.global_step,
            )

    if best_path.exists():
        load_checkpoint(
            best_path,
            trainer.model,
            map_location="cpu",
            strict=True,
            migrate_history=True,
        )
        log("Restored best joint checkpoint.")

    return {
        "best_path": best_path
        if best_path.exists()
        else None,
        "best_row": best_row,
        "model": trainer.model,
    }


## 7. Preflight

The controller refuses to begin expensive work if:
- CUDA is unavailable;
- the exact source checkpoint is missing;
- the data scene is wrong;
- disk headroom is critically low;
- the strict spatial-only forward cannot execute.

A modest disk warning does **not** abort; only critically low space does.


In [ ]:
def initialize_run() -> tuple[StirNet, Any]:
    global RUN_DIR
    global LOG_PATH
    global EVENTS_PATH
    global TRAIN_METRICS_PATH
    global SNAPSHOT_METRICS_PATH
    global ERRORS_PATH
    global STATE_PATH

    global batch_cpu
    global b
    global targets
    global target
    global current_labels_native
    global gt_labels_native
    global spacing_native
    global dref_um
    global gt_ids
    global gt_centers_cellscale
    global source9_gt_ids
    global source9_gt_id_set
    global source9_gt_indices
    global source9_gt_centers
    global GT_FOREGROUND
    global GT_INTERNAL
    global GT_ALL_BOUNDARY
    global SOURCE9_NATIVE

    if device.type != "cuda":
        raise RuntimeError(
            "Notebook 27 requires CUDA."
        )

    if not STEP30_CHECKPOINT.exists():
        raise FileNotFoundError(
            f"Step-30 checkpoint not found: {STEP30_CHECKPOINT}"
        )

    timestamp = time.strftime(
        "%Y%m%d_%H%M%S"
    )
    RUN_DIR = (
        RUN_PARENT
        / f"27_overnight_{timestamp}"
    )
    RUN_DIR.mkdir(
        parents=True,
        exist_ok=False,
    )

    LOG_PATH = RUN_DIR / "overnight.log"
    EVENTS_PATH = RUN_DIR / "events.jsonl"
    TRAIN_METRICS_PATH = (
        RUN_DIR / "training_metrics.jsonl"
    )
    SNAPSHOT_METRICS_PATH = (
        RUN_DIR / "snapshot_metrics.jsonl"
    )
    ERRORS_PATH = RUN_DIR / "errors.jsonl"
    STATE_PATH = RUN_DIR / "run_state.json"

    RUN_CLOCK["start"] = time.time()
    RUN_CLOCK["hard_deadline"] = (
        RUN_CLOCK["start"]
        + TOTAL_BUDGET_HOURS * 3600
    )
    RUN_CLOCK["training_deadline"] = (
        RUN_CLOCK["hard_deadline"]
        - EVALUATION_RESERVE_MINUTES * 60
    )

    log(
        "Notebook 27 overnight run starting."
    )
    log(f"Run directory: {RUN_DIR}")
    log(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )
    log(
        f"Free disk: {free_disk_gib(REPO_ROOT):.2f} GiB"
    )

    free_gib = free_disk_gib(REPO_ROOT)
    if free_gib < 2.5:
        raise RuntimeError(
            "Less than 2.5 GiB free on the repository drive. "
            "Refusing an unattended overnight run."
        )
    if free_gib < 6.0:
        log(
            "WARNING: disk headroom is below 6 GiB. "
            "Checkpoint policy remains model-only and bounded."
        )

    # Record git state without requiring a clean tree.
    try:
        git_head = subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=REPO_ROOT,
            text=True,
        ).strip()
        git_status = subprocess.check_output(
            ["git", "status", "--short"],
            cwd=REPO_ROOT,
            text=True,
        ).strip()
    except Exception as exc:
        git_head = "unknown"
        git_status = f"git inspection failed: {exc}"

    record_event(
        "repository",
        git_head=git_head,
        git_status=git_status,
    )

    batch_cpu, sample_info = build_real_batch(
        DATA_DIR
    )

    b = move_batch_to_device(
        batch_cpu,
        device,
    )
    b["spatial_inputs"] = b[
        "spatial_inputs"
    ].to(dtype=AMP_DTYPE)
    b["instance_labels"] = b[
        "instance_labels"
    ].to(dtype=torch.int32)

    targets = batch_cpu["targets"]
    target = targets[0]

    current_labels_native = (
        batch_cpu["instance_labels"][0]
        .detach()
        .cpu()
        .numpy()
        .astype(np.int32, copy=False)
    )

    gt_labels_native = (
        torch.as_tensor(
            target["label_map"]
        )
        .detach()
        .cpu()
        .numpy()
        .astype(np.int32, copy=False)
    )

    spacing_native = (
        batch_cpu["spacing_um"][0]
        .detach()
        .cpu()
        .numpy()
        .astype(np.float64)
    )

    dref_um = float(
        batch_cpu["dref_um"][0]
    )

    gt_ids = (
        target_ids(target)
        .detach()
        .cpu()
        .long()
    )

    gt_centers_cellscale = torch.as_tensor(
        target["centers_cellscale"],
        dtype=torch.float32,
    )

    source9_gt_ids = np.unique(
        gt_labels_native[
            current_labels_native
            == SOURCE_ID
        ]
    )
    source9_gt_ids = source9_gt_ids[
        source9_gt_ids > 0
    ].astype(int)

    source9_gt_id_set = set(
        source9_gt_ids.tolist()
    )

    source9_gt_indices = torch.tensor(
        [
            idx
            for idx, gt_id
            in enumerate(gt_ids.tolist())
            if int(gt_id)
            in source9_gt_id_set
        ],
        dtype=torch.long,
    )

    source9_gt_centers = (
        gt_centers_cellscale[
            source9_gt_indices
        ].float()
    )

    if len(source9_gt_ids) != 9:
        raise RuntimeError(
            "Source-9 invariant changed: "
            f"expected 9 GT cells, got {len(source9_gt_ids)}."
        )

    GT_FOREGROUND = torch.as_tensor(
        target["foreground"]
    ).detach().cpu().bool()

    GT_INTERNAL = (
        torch.as_tensor(
            target["internal_boundary"]
        )
        .detach()
        .cpu()
        > 0.5
    )

    GT_ALL_BOUNDARY = (
        torch.as_tensor(
            target["boundary"]
        )
        .detach()
        .cpu()
        > 0.5
    )

    SOURCE9_NATIVE = torch.from_numpy(
        current_labels_native == SOURCE_ID
    )

    setup_source9_crop()

    model, cfg, info = load_fresh_model(
        STEP30_CHECKPOINT
    )

    if int(info.get("step", -1)) != 30:
        raise RuntimeError(
            "Expected the source checkpoint to be step 30; "
            f"got {info.get('step')}."
        )

    atomic_model_checkpoint(
        RUN_DIR / "checkpoint_start.pt",
        model=model,
        cfg=cfg,
        step=30,
        extra={
            "notebook": 27,
            "stage": "start",
            "source_checkpoint": str(
                STEP30_CHECKPOINT
            ),
        },
    )

    update_state(
        status="preflight",
        git_head=git_head,
        source_checkpoint=str(
            STEP30_CHECKPOINT
        ),
        source9_gt_ids=source9_gt_ids.tolist(),
        budget_hours=TOTAL_BUDGET_HOURS,
        evaluation_reserve_minutes=(
            EVALUATION_RESERVE_MINUTES
        ),
    )

    # Forward-only sanity check.
    model.eval()
    with torch.no_grad(), torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        outputs = forward_spatial_only_full(
            model
        )

    baseline_row, baseline_per_cell = (
        evaluate_outputs(
            outputs,
            label="preflight_step30",
            stage="preflight",
            step=30,
        )
    )

    baseline_per_cell.to_csv(
        RUN_DIR / "preflight_per_cell.csv",
        index=False,
    )

    log(
        "Preflight spatial forward OK | "
        f"proposal recall0.5="
        f"{baseline_row['source9_recall_0p5']:.3f} | "
        f"source9 proposals="
        f"{baseline_row['source9_associated_learned']}"
    )

    del outputs
    safe_cuda_cleanup()

    update_state(
        status="preflight_passed",
    )

    return model, cfg


# 8. Overnight controller

This is the only heavy cell.

The controller:
1. initializes and preflights;
2. trains proposals;
3. runs the anchor-vs-query ablation;
4. trains spatial-only queries;
5. trains native masks;
6. spends the remaining training budget on the normal full joint model;
7. evaluates every surviving best checkpoint;
8. writes an automatic diagnosis.

If any long stage fails, the traceback is written to `errors.jsonl`, earlier checkpoints/logs remain safe, and the controller continues where possible.


In [ ]:
def load_model_from_path(
    checkpoint: Path,
) -> tuple[StirNet, Any]:
    model, cfg, _ = load_fresh_model(
        checkpoint
    )
    return model, cfg


def final_evaluate_checkpoint(
    checkpoint: Path,
    *,
    label: str,
    mode: str,
    include_native: bool,
    save_visual: bool = False,
) -> dict[str, Any]:
    log(
        f"Final evaluation: {label} ({checkpoint.name})"
    )

    model, cfg = load_model_from_path(
        checkpoint
    )

    if mode == "joint":
        outputs = forward_full_model(model)
    elif mode == "spatial":
        model.eval()
        with torch.no_grad(), torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            outputs = forward_spatial_only_full(
                model
            )
    else:
        raise ValueError(mode)

    row, per_cell = evaluate_outputs(
        outputs,
        label=label,
        stage="final_evaluation",
        step=int(
            load_checkpoint(
                checkpoint,
                model,
                map_location="cpu",
                strict=True,
                migrate_history=True,
            ).get("step", -1)
        ),
    )

    per_cell.to_csv(
        RUN_DIR
        / f"final_{label}_per_cell.csv",
        index=False,
    )

    if include_native:
        visual_path = (
            RUN_DIR
            / f"napari_{label}_compact.npz"
            if save_visual
            else None
        )

        native_row, native_per_cell = (
            native_source9_metrics(
                outputs,
                save_compact_npz=visual_path,
            )
        )

        row.update(native_row)

        native_per_cell.to_csv(
            RUN_DIR
            / f"final_{label}_native_per_cell.csv",
            index=False,
        )

    append_jsonl(
        SNAPSHOT_METRICS_PATH,
        {
            **row,
            "final_checkpoint": str(
                checkpoint
            ),
        },
    )

    del outputs, model
    safe_cuda_cleanup()

    return row


def interpret_ablation(
    ablation_df: pd.DataFrame,
) -> dict[str, Any]:
    if not len(ablation_df):
        return {
            "ablation_interpretation":
            "No anchor/query ablation completed."
        }

    indexed = ablation_df.set_index(
        "ablation"
    )

    required = {
        "normal",
        "identical_query",
        "collapsed_anchor",
        "shuffled_query",
    }
    if not required.issubset(
        set(indexed.index)
    ):
        return {
            "ablation_interpretation":
            "Ablation incomplete; inspect anchor_query_ablation.csv."
        }

    def dice(name):
        return float(
            indexed.loc[
                name,
                "source9_local_soft_dice_mean",
            ]
        )

    normal = dice("normal")
    identical = dice("identical_query")
    collapsed = dice("collapsed_anchor")
    shuffled = dice("shuffled_query")

    result = {
        "ablation_normal_dice": normal,
        "ablation_identical_query_dice": identical,
        "ablation_collapsed_anchor_dice": collapsed,
        "ablation_shuffled_query_dice": shuffled,
    }

    if (
        math.isfinite(normal)
        and math.isfinite(identical)
        and abs(identical - normal)
        <= max(0.02, 0.10 * abs(normal))
    ):
        query_note = (
            "Making source-9 query embeddings identical had little effect; "
            "high query cosine is probably not the primary failure."
        )
    else:
        query_note = (
            "Making source-9 query embeddings identical changed mask quality "
            "substantially; query content contributes materially."
        )

    if (
        math.isfinite(normal)
        and math.isfinite(collapsed)
        and collapsed < normal - max(
            0.02,
            0.10 * abs(normal),
        )
    ):
        anchor_note = (
            "Collapsing anchors harmed masks; proposal geometry is an "
            "important instance-identity signal."
        )
    else:
        anchor_note = (
            "Collapsing anchors did not clearly harm masks; the current mask "
            "path may not be exploiting anchors strongly enough."
        )

    result[
        "ablation_interpretation"
    ] = query_note + " " + anchor_note

    return result


def automatic_diagnosis(
    final_rows: list[dict[str, Any]],
    ablation_df: pd.DataFrame,
) -> dict[str, Any]:
    by_label = {
        row["label"]: row
        for row in final_rows
    }

    proposal = by_label.get(
        "best_proposal",
        {},
    )
    query = by_label.get(
        "best_spatial_query",
        {},
    )
    native = by_label.get(
        "best_native",
        {},
    )
    joint = by_label.get(
        "best_joint",
        {},
    )

    proposal_good = (
        proposal.get(
            "source9_recall_0p5",
            0.0,
        ) >= 8 / 9
        and proposal.get(
            "source9_recall_1p0",
            0.0,
        ) >= 8 / 9
    )

    query_dice = query.get(
        "source9_local_soft_dice_mean",
        float("nan"),
    )

    native_dice = native.get(
        "native_soft_dice_mean",
        float("nan"),
    )

    if not proposal_good:
        verdict = (
            "RED_PROPOSAL_GEOMETRY_STILL_UNRELIABLE"
        )
        next_action = (
            "Proposal recall/cardinality is still the first bottleneck. "
            "Do not redesign the mask decoder first."
        )

    elif (
        query_dice is None
        or not math.isfinite(
            float(query_dice)
        )
        or float(query_dice) < 0.25
    ):
        verdict = (
            "YELLOW_GOOD_PROPOSALS_POOR_QUERY_MASKS"
        )
        next_action = (
            "The spatial proposal mechanism works, but support-aligned coarse "
            "instance masks remain poor after substantial training. Use the "
            "anchor-vs-query ablation to choose between query-content fixes "
            "and a more explicitly anchor-relative mask decoder."
        )

    elif (
        native_dice is not None
        and math.isfinite(
            float(native_dice)
        )
        and float(native_dice)
        < 0.8 * float(query_dice)
    ):
        verdict = (
            "YELLOW_COARSE_WORKS_NATIVE_RENDERING_BOTTLENECK"
        )
        next_action = (
            "Coarse query masks are useful, but native rendering loses them. "
            "Focus on native mask features/support/rendering."
        )

    else:
        verdict = (
            "GREEN_NOTEBOOK24_PATH_TRAINS_USEFULLY"
        )
        next_action = (
            "The proposal-to-mask path is learning useful source-9 instances. "
            "Continue with broader corruption/generalization tests before "
            "another architecture rewrite."
        )

    result = {
        "verdict": verdict,
        "next_action": next_action,
        "proposal_good": proposal_good,
        "best_query_local_soft_dice": _jsonable(
            query_dice
        ),
        "best_native_soft_dice": _jsonable(
            native_dice
        ),
        "best_joint_local_soft_dice": _jsonable(
            joint.get(
                "source9_local_soft_dice_mean",
                float("nan"),
            )
        ),
    }

    result.update(
        interpret_ablation(
            ablation_df
        )
    )

    return result


def run_overnight() -> dict[str, Any]:
    model = None
    cfg = None

    stage_results: dict[str, Any] = {}
    ablation_df = pd.DataFrame()
    final_rows: list[dict[str, Any]] = []

    try:
        model, cfg = initialize_run()

        # -------------------------------------------------------------
        # Stage 1: proposal warm-up
        # -------------------------------------------------------------
        proposal_deadline = min(
            RUN_CLOCK["training_deadline"],
            time.time()
            + PROPOSAL_MAX_MINUTES * 60,
        )

        try:
            stage_results["proposal"] = (
                run_proposal_stage(
                    model,
                    cfg,
                    stage_deadline=proposal_deadline,
                )
            )
        except Exception as exc:
            record_error(
                "proposal_stage",
                exc,
            )
            safe_cuda_cleanup()

        # -------------------------------------------------------------
        # Causal anchor/query ablation
        # -------------------------------------------------------------
        try:
            ablation_df = (
                run_anchor_query_ablation(
                    model
                )
            )
        except Exception as exc:
            record_error(
                "anchor_query_ablation",
                exc,
            )
            safe_cuda_cleanup()

        # -------------------------------------------------------------
        # Stage 2: spatial-only query training
        # -------------------------------------------------------------
        proposal_best = (
            stage_results.get(
                "proposal",
                {},
            ).get("best_row")
        )

        proposal_gate = bool(
            proposal_best
            and proposal_best.get(
                "source9_recall_1p0",
                0.0,
            )
            >= MIN_SOURCE9_RECALL_1DREF
        )

        update_state(
            proposal_gate=proposal_gate
        )

        if (
            proposal_gate
            and training_time_left_seconds() > 0
        ):
            query_deadline = min(
                RUN_CLOCK["training_deadline"],
                time.time()
                + SPATIAL_QUERY_MAX_MINUTES
                * 60,
            )

            try:
                stage_results[
                    "spatial_query"
                ] = run_spatial_query_stage(
                    model,
                    cfg,
                    stage_deadline=query_deadline,
                )
            except Exception as exc:
                record_error(
                    "spatial_query_stage",
                    exc,
                )
                safe_cuda_cleanup()
        else:
            log(
                "Spatial-query stage skipped because proposal gate failed "
                "or training time is exhausted."
            )

        # -------------------------------------------------------------
        # Stage 3: spatial-only native training
        # -------------------------------------------------------------
        if (
            training_time_left_seconds() > 0
            and (
                RUN_DIR
                / "checkpoint_best_spatial_query.pt"
            ).exists()
        ):
            native_deadline = min(
                RUN_CLOCK["training_deadline"],
                time.time()
                + SPATIAL_NATIVE_MAX_MINUTES
                * 60,
            )

            try:
                stage_results[
                    "spatial_native"
                ] = run_spatial_native_stage(
                    model,
                    cfg,
                    stage_deadline=native_deadline,
                )
            except Exception as exc:
                record_error(
                    "spatial_native_stage",
                    exc,
                )
                safe_cuda_cleanup()
        else:
            log(
                "Spatial-native stage skipped."
            )

        # -------------------------------------------------------------
        # Stage 4: normal full-model joint training with remaining time
        # -------------------------------------------------------------
        if training_time_left_seconds() > 0:
            try:
                stage_results[
                    "joint"
                ] = run_joint_stage(
                    model,
                    cfg,
                    stage_deadline=(
                        RUN_CLOCK[
                            "training_deadline"
                        ]
                    ),
                )

                if stage_results[
                    "joint"
                ].get("model") is not None:
                    model = stage_results[
                        "joint"
                    ]["model"]

            except Exception as exc:
                record_error(
                    "joint_stage",
                    exc,
                )
                safe_cuda_cleanup()
        else:
            log(
                "Joint stage skipped: evaluation reserve reached."
            )

    except Exception as exc:
        # Preflight or unexpected controller-level failure.
        if RUN_DIR is not None:
            record_error(
                "controller",
                exc,
            )
        else:
            print(
                "Controller failed before run directory initialization:",
                repr(exc),
                flush=True,
            )

    finally:
        # -------------------------------------------------------------
        # Final evaluation always runs on every surviving best checkpoint.
        # -------------------------------------------------------------
        if RUN_DIR is None:
            return {
                "status": "failed_before_run_initialization"
            }

        update_state(
            status="final_evaluation"
        )

        log(
            "=== FINAL EVALUATION OF SURVIVING CHECKPOINTS ==="
        )

        candidates = [
            (
                "start",
                RUN_DIR / "checkpoint_start.pt",
                "spatial",
                False,
            ),
            (
                "best_proposal",
                RUN_DIR
                / "checkpoint_best_proposal.pt",
                "spatial",
                False,
            ),
            (
                "best_spatial_query",
                RUN_DIR
                / "checkpoint_best_spatial_query.pt",
                "spatial",
                False,
            ),
            (
                "best_native",
                RUN_DIR
                / "checkpoint_best_native.pt",
                "spatial",
                True,
            ),
            (
                "best_joint",
                RUN_DIR
                / "checkpoint_best_joint.pt",
                "joint",
                True,
            ),
        ]

        existing = [
            item
            for item in candidates
            if item[1].exists()
        ]

        for index, (
            label,
            checkpoint,
            mode,
            include_native,
        ) in enumerate(existing):
            if total_time_left_seconds() <= 60:
                log(
                    "Hard deadline is imminent; stopping final checkpoint sweep."
                )
                break

            try:
                save_visual = bool(
                    SAVE_COMPACT_VISUAL_ARTIFACT
                    and include_native
                    and index
                    == len(existing) - 1
                )

                row = final_evaluate_checkpoint(
                    checkpoint,
                    label=label,
                    mode=mode,
                    include_native=include_native,
                    save_visual=save_visual,
                )
                final_rows.append(row)

            except Exception as exc:
                record_error(
                    f"final_evaluation/{label}",
                    exc,
                )
                safe_cuda_cleanup()

        final_df = pd.DataFrame(
            final_rows
        )

        if len(final_df):
            final_df.to_csv(
                RUN_DIR
                / "final_checkpoint_comparison.csv",
                index=False,
            )

        diagnosis = automatic_diagnosis(
            final_rows,
            ablation_df,
        )

        summary = {
            "notebook": 27,
            "status": "completed_controller",
            "started_at": time.strftime(
                "%Y-%m-%d %H:%M:%S",
                time.localtime(
                    RUN_CLOCK["start"]
                ),
            ),
            "finished_at": now_text(),
            "elapsed_hours": elapsed_hours(),
            "free_disk_gib": free_disk_gib(
                REPO_ROOT
            ),
            "checkpoint_disk_gib": (
                checkpoint_file_size_gib()
            ),
            "run_dir": str(RUN_DIR),
            "source_checkpoint": str(
                STEP30_CHECKPOINT
            ),
            "source9_gt_ids": (
                source9_gt_ids.tolist()
                if source9_gt_ids
                is not None
                else []
            ),
            "final_checkpoints_evaluated": [
                row["label"]
                for row in final_rows
            ],
            **diagnosis,
        }

        atomic_json(
            RUN_DIR / "summary.json",
            summary,
        )

        update_state(
            status="finished",
            verdict=diagnosis[
                "verdict"
            ],
        )

        # The rolling recovery file is redundant once at least one best
        # checkpoint survived. Delete it to recover disk.
        best_files = list(
            RUN_DIR.glob(
                "checkpoint_best_*.pt"
            )
        )
        recovery = (
            RUN_DIR
            / "checkpoint_recovery.pt"
        )
        if best_files and recovery.exists():
            try:
                recovery.unlink()
                log(
                    "Deleted rolling recovery checkpoint after successful best-checkpoint saves."
                )
            except Exception as exc:
                record_error(
                    "cleanup_recovery",
                    exc,
                )

        log("=" * 88)
        log(
            "NOTEBOOK 27 OVERNIGHT RUN FINISHED"
        )
        log(
            f"Verdict: {diagnosis['verdict']}"
        )
        log(
            f"Next action: {diagnosis['next_action']}"
        )
        log(
            f"Elapsed: {elapsed_hours():.2f} h"
        )
        log(
            f"Final free disk: {free_disk_gib(REPO_ROOT):.2f} GiB"
        )
        log("=" * 88)

        return summary


overnight_summary = run_overnight()

print()
print(
    json.dumps(
        overnight_summary,
        indent=2,
        default=_jsonable,
    )
)


# 9. Morning inspection

When the run finishes, the most useful files are:

```text
summary.json
final_checkpoint_comparison.csv
anchor_query_ablation.csv
best_proposal_per_gt.csv
best_spatial_query_per_cell.csv
best_native_coarse_per_cell.csv
best_joint_per_cell.csv
overnight.log
errors.jsonl               # only if something failed
```

If the last native/joint evaluation succeeded, one small file such as:

```text
napari_best_joint_compact.npz
```

contains only a source-9 crop, GT/current labels, max mask probability, and combined predicted labels.

No full-volume per-query predictions are saved.

## How to interpret the morning verdict

### `RED_PROPOSAL_GEOMETRY_STILL_UNRELIABLE`
The proposal mechanism itself is still the first bottleneck.

### `YELLOW_GOOD_PROPOSALS_POOR_QUERY_MASKS`
The model finds the cells, but the current query/mask pathway cannot exploit the anchors well enough even after substantial optimization.

Use `anchor_query_ablation.csv`:
- identical queries ≈ normal, but collapsed anchors much worse → geometry matters; make mask decoding more explicitly anchor-relative;
- identical queries much worse → query content matters; preserve/improve proposal-local query content;
- collapsed anchors ≈ normal → decoder is not using anchor geometry strongly enough.

### `YELLOW_COARSE_WORKS_NATIVE_RENDERING_BOTTLENECK`
Coarse masks became useful but native rendering lost quality.

### `GREEN_NOTEBOOK24_PATH_TRAINS_USEFULLY`
The new proposal architecture is learning usable instance decomposition; do not rewrite it before broader tests.


In [ ]:
# Lightweight morning convenience cell.
# It never starts training again.

if RUN_DIR is not None:
    summary_path = RUN_DIR / "summary.json"

    if summary_path.exists():
        morning_summary = json.loads(
            summary_path.read_text(
                encoding="utf-8"
            )
        )
        display(
            pd.DataFrame(
                [morning_summary]
            ).T
        )

    comparison_path = (
        RUN_DIR
        / "final_checkpoint_comparison.csv"
    )

    if comparison_path.exists():
        morning_comparison = pd.read_csv(
            comparison_path
        )

        preferred_columns = [
            "label",
            "source9_associated_learned",
            "source9_recall_0p5",
            "source9_recall_1p0",
            "source9_gt_with_duplicate_0p5",
            "source9_gt_matched_proposal_query",
            "source9_local_soft_dice_mean",
            "source9_local_hard_dice_mean",
            "source9_pairwise_coarse_soft_dice",
            "native_soft_dice_mean",
            "foreground_hard_dice",
        ]

        display(
            morning_comparison[
                [
                    c
                    for c
                    in preferred_columns
                    if c in morning_comparison.columns
                ]
            ]
        )
